# ManufactureIQ — wafer-map CNN on a GPU (Colab)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Prakruthi19/manufactureiq/blob/main/notebooks/wafer_cnn_colab.ipynb)

Runs the GPU-heavy experiments that are too slow on a laptop CPU. All the logic lives in `scripts/`; this notebook only calls them in order, so every number can be reproduced locally with the same commands.

| Step | Script | What it answers |
|---|---|---|
| 1 | `download_data.py`, `build_wafer_features.py` | the 172,950 labelled wafers, same 48×48 maps and lot-aware split as on the laptop |
| 2 | `train_wafer_cnn.py --tag scratch` | CNN from random weights (re-run of the laptop result, on GPU) |
| 3 | `download_unlabelled.py` | the ~638k wafers nobody labelled |
| 4 | `pretrain_wafer_cnn.py` | self-supervised pre-training (SimCLR) on them — no labels used |
| 5 | `train_wafer_cnn.py --init … --tag pretrained` | does pre-training beat random weights, and XGBoost (0.861)? |
| 6 | `--label-fraction 0.1` | does it help more when only 10% of the labels exist? |

**Before you start:** Runtime → Change runtime type → **T4 GPU**. Total ≈ 1.5–2 h on a free T4.

Stuck on a cell? Click the ✦ Gemini button and ask it to explain the cell or an error.

## 0 · GPU check and code

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch; assert torch.cuda.is_available(), "No GPU: Runtime → Change runtime type → T4 GPU"

In [ ]:
REPO_URL = "https://github.com/Prakruthi19/manufactureiq.git"   # your fork / repo
BRANCH   = "main"

# Model .pkl files are in Git LFS and not needed here — skip downloading them.
!GIT_LFS_SKIP_SMUDGE=1 git clone -q --depth 1 -b {BRANCH} {REPO_URL} /content/manufactureiq
%cd /content/manufactureiq
!pip -q install imbalanced-learn
!git log -1 --oneline

## 1 · Cache on Google Drive (optional, recommended)

Colab wipes `/content` when the session ends. This keeps the processed data and results in `My Drive/manufactureiq/` so a second session skips the downloads and the feature build.

Shortcut: upload your laptop's `data/processed/wm811k_features.npz` and `wm811k_maps48.npz` to `My Drive/manufactureiq/processed/` and step 2 takes seconds instead of ~30 min.

In [ ]:
USE_DRIVE = True

import os, shutil, glob
CACHE = None
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    CACHE = "/content/drive/MyDrive/manufactureiq"
    os.makedirs(f"{CACHE}/processed", exist_ok=True)
    os.makedirs("data/processed", exist_ok=True)
    for f in glob.glob(f"{CACHE}/processed/*.npz"):
        shutil.copy(f, "data/processed/")
        print("restored", os.path.basename(f))

def save_to_drive(*paths):
    if CACHE:
        for p in paths:
            for f in glob.glob(p):
                dest = f"{CACHE}/processed" if f.startswith("data/") else f"{CACHE}/models"
                os.makedirs(dest, exist_ok=True)
                shutil.copy(f, dest)

## 2 · Labelled wafers → features + 48×48 maps

In [ ]:
import os
if os.path.exists("data/processed/wm811k_maps48.npz") and os.path.exists("data/processed/wm811k_features.npz"):
    print("Already built — skipping")
else:
    !python -c "from scripts.download_data import download_wm811k; download_wm811k(workers=32)"
    !python scripts/build_wafer_features.py
    save_to_drive("data/processed/wm811k_features.npz", "data/processed/wm811k_maps48.npz")

## 3 · Baseline: CNN from random weights (≈ 5 min on GPU)

In [ ]:
!python scripts/train_wafer_cnn.py --tag scratch
save_to_drive("models/wafer_cnn_scratch*")

## 4 · Unlabelled wafers + self-supervised pre-training

The download is ~1.3 GB of compressed JSON (≈ 5–10 min). Wafers from validation or test lot-blocks are dropped inside `pretrain_wafer_cnn.py`, so the test lots stay unseen.

Watch the **kNN val F1** column: it uses labels only to *monitor*. If it rises across epochs, the conv layers are learning defect structure without ever seeing a label.

In [ ]:
if os.path.exists("data/processed/wm811k_unlabelled_maps48.npz"):
    print("Already downloaded — skipping")
else:
    !python scripts/download_unlabelled.py
    save_to_drive("data/processed/wm811k_unlabelled_maps48.npz")

In [ ]:
PRETRAIN_EPOCHS = 30     # ~1 min each on a T4
!python scripts/pretrain_wafer_cnn.py --epochs {PRETRAIN_EPOCHS}
save_to_drive("models/wafer_cnn_encoder_simclr*")

## 5 · Fine-tune the pre-trained CNN on the labels

In [ ]:
!python scripts/train_wafer_cnn.py --init models/wafer_cnn_encoder_simclr.pt --tag pretrained
save_to_drive("models/wafer_cnn_pretrained*")

## 6 · Labels-scarce test: only 10% of the training lots

Pre-training matters most when labels are expensive (a new product, a new fab). Same split, same validation and test lots; only the labelled training wafers shrink.

In [ ]:
!python scripts/train_wafer_cnn.py --label-fraction 0.1 --tag scratch_10pct
!python scripts/train_wafer_cnn.py --label-fraction 0.1 --init models/wafer_cnn_encoder_simclr.pt --tag pretrained_10pct
save_to_drive("models/wafer_cnn_*10pct*")

## 7 · Results

In [ ]:
import json, pandas as pd
rows = {}
xgb = json.load(open("models/wafer_classifier_metrics.json"))
rows["XGBoost, 26 features (laptop)"] = {**xgb["test"], "train_wafers": xgb["split"]["train"]}
for tag, name in [("scratch", "CNN, random init"), ("pretrained", "CNN, pre-trained"),
                  ("scratch_10pct", "CNN, random init, 10% labels"), ("pretrained_10pct", "CNN, pre-trained, 10% labels")]:
    p = f"models/wafer_cnn_{tag}_metrics.json"
    if os.path.exists(p):
        m = json.load(open(p))
        rows[name] = {**m["test"], "train_wafers": m["train_wafers"], "val_macro_f1": m["validation"]["macro_f1"]}
cols = ["val_macro_f1", "macro_f1", "macro_f1_patterns", "defect_recall", "false_alarm_rate", "train_wafers"]
pd.DataFrame(rows).T.reindex(columns=cols).round(3)

Reading it honestly:
* Compare CNNs with each other on **validation** too — XGBoost's test model was refit on train+validation, the CNNs were not.
* The pre-trained model is only better if it wins on validation *and* test; one run is one seed, so treat gaps under ~0.01 as noise.

## 8 · Take the results back to the laptop

In [ ]:
!zip -q -j cnn_results.zip models/wafer_cnn_*_metrics.json models/wafer_cnn_*.pt
from google.colab import files
files.download("cnn_results.zip")

Unzip into the repo's `models/` folder on your laptop. The metrics files are what the docs and interview guide quote.